In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print (device)

cuda


<hr>
<hr>

In [ ]:
quant = BitsAndBytesConfig (
    load_in_4bit = True,  #Quantização
    bnb_4bit_quant_type = 'nf4',  #Tipo de Quantização
    bnb_4bit_use_double_quant = True, #Double Quantization
    bnb_4bit_compute_dtype = 'float16', #Tipo de precisão usada nos cálculos durante a inferência.
)


modelo = AutoModelForCausalLM.from_pretrained (r"microsoft/phi-4", quantization_config = quant, device_map = device)
tokenizer = AutoTokenizer.from_pretrained (r"microsoft/phi-4")

modelo.save_pretrained (r"C:\Users\Admin\Desktop\models\microsoftphi-4-Q4")
tokenizer.save_pretrained (r"C:\Users\Admin\Desktop\models\microsoftphi-4-Q4")

<hr>
<hr>

In [3]:
path = r"C:\Users\Admin\Desktop\models\microsoftphi-4-Q4"

model = AutoModelForCausalLM.from_pretrained (path, device_map = device)
tokenizer = AutoTokenizer.from_pretrained (path)

Loading weights: 100%|██████████| 243/243 [00:15<00:00, 15.97it/s]


In [ ]:
prompt = "Olá, quem foi o primeiro rei de Portugal ?"

tokens = tokenizer (prompt, return_tensors = "pt").to (device)

with torch.no_grad():
    logits = model.generate (**tokens, max_new_tokens = 512)

input_length = tokens["input_ids"].shape[1]
response_tokens = logits[0][input_length:]

output = tokenizer.decode(response_tokens, skip_special_tokens = True)

<hr>
<hr>

In [4]:
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

from pathlib import Path
from langchain_core.documents import Document


In [5]:
path = Path (r"C:\Users\Admin\Desktop\ip\How To RAG\docs")

documentos = []
chunk_id = 1

for docs in path.iterdir():

    file_dtype = docs.suffix.lower()

    if file_dtype == ".pdf":

        parsing = partition_pdf (str(docs), languages = ["por"])
        chunks = chunk_by_title (parsing[3:], max_characters = 750)

    '''elif file_dtype == ".txt":

        parsing = partition_text (str(docs), languages = ["por"])
        chunks = chunk_elements (parsing, max_characters = 750)

    elif file_dtype == ".docx":        
        
        parsing = partition_docx (str(docs), languages = ["por"])
        chunks = chunk_by_title (parsing, max_characters = 750)'''


    for chunk in chunks:

        documentos.append (
            Document (
                page_content = chunk.text,
                metadata = {
                    "title": docs.name,
                    "chunk_id": chunk_id
                }
            )
        )

        chunk_id += 1


#documentos

<hr>
<hr>

420
